# Cleaning the OpenFlights Dataset

This notebook cleans selected OpenFlights `.dat` files and prepares them for loading into a relational SQL database.

The final database is designed around the following ERD tables. This is based off of the schema built in Mermaid. Hopefully, if required, the schema can be updated and this notebook can be edited accordingly to reflect the necessary changes:

- `Country`
- `Airport`
- `Airline`
- `Plane`
- `Route`
- `RouteEquipment`

The main cleaning tasks are:

1. Add column names to the raw `.dat` files.
2. Convert OpenFlights missing values, written as `\N`, into proper null values.
3. Map country names to ISO country codes.
4. Clean airport, airline, plane and route records.
5. Create a surrogate `route_id` for routes.
6. Split the route `equipment` field into a proper many-to-many bridge table.
7. Check that foreign key relationships are valid.
8. Export one cleaned CSV file per table.


## 1. Expected project structure

Before running the notebook, place the OpenFlights `.dat` files inside a folder called `raw` (this is for portability and should reduce file structure headache).

The file structure should look like this:

```text
openflights_project/
├── openflights_cleaning_notebook.ipynb
├── raw/
│   ├── countries.dat
│   ├── airports.dat
│   ├── airlines.dat
│   ├── planes.dat
│   └── routes.dat
└── cleaned/
```

The `cleaned` folder will be created automatically if it does not already exist.


In [3]:
import pandas as pd
from pathlib import Path

RAW = Path("raw")
CLEANED = Path("cleaned")
CLEANED.mkdir(exist_ok=True)

NULL_VALUES = ["\\N", ""]

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)


## 2. Define column names

The OpenFlights `.dat` files are comma-delimited files without headers.  
So we define the column names ourselves before loading the data.

These column names are based on the OpenFlights data documentation and are aligned with the ERD tables we want to build.


In [4]:
country_cols = [
    "name",
    "iso_code",
    "dafif_code",
]

airport_cols = [
    "airport_id",
    "name",
    "city",
    "country_name",
    "iata_code",
    "icao_code",
    "latitude",
    "longitude",
    "altitude",
    "timezone_offset",
    "dst",
    "tz_database_timezone",
    "type",
    "source",
]

airline_cols = [
    "airline_id",
    "name",
    "alias",
    "iata_code",
    "icao_code",
    "callsign",
    "country_name",
    "active",
]

plane_cols = [
    "name",
    "iata_code",
    "icao_code",
]

route_cols = [
    "airline_code",
    "airline_id",
    "source_airport_code",
    "source_airport_id",
    "destination_airport_code",
    "destination_airport_id",
    "codeshare",
    "stops",
    "equipment",
]


## 3. Load the raw `.dat` files

OpenFlights uses `\N` to represent missing values.  
When reading the files, we tell pandas to treat `\N` and empty strings as null values.


In [5]:
def read_dat(filename, columns):
    """Read an OpenFlights .dat file as a headerless CSV file."""
    return pd.read_csv(
        RAW / filename,
        header=None,
        names=columns,
        na_values=NULL_VALUES,
        keep_default_na=True,
        encoding="utf-8",
    )


countries_raw = read_dat("countries.dat", country_cols)
airports_raw = read_dat("airports.dat", airport_cols)
airlines_raw = read_dat("airlines.dat", airline_cols)
planes_raw = read_dat("planes.dat", plane_cols)
routes_raw = read_dat("routes.dat", route_cols)

print("Raw files loaded.")
print(f"Countries: {len(countries_raw)}")
print(f"Airports: {len(airports_raw)}")
print(f"Airlines: {len(airlines_raw)}")
print(f"Planes: {len(planes_raw)}")
print(f"Routes: {len(routes_raw)}")


Raw files loaded.
Countries: 261
Airports: 7698
Airlines: 6162
Planes: 246
Routes: 67663


## 4. Inspect the raw data

This section gives a quick look at the raw data before cleaning.


In [6]:
countries_raw.head()


,name,iso_code,dafif_code
0,"Bonaire, Saint Eustatius and Saba",BQ,NaN
1,Aruba,AW,AA
2,Antigua and Barbuda,AG,AC
3,United Arab Emirates,AE,AE
4,Afghanistan,AF,AF


In [7]:
airports_raw.head()


,airport_id,name,city,country_name,iata_code,icao_code,latitude,longitude,altitude,timezone_offset,dst,tz_database_timezone,type,source
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,-6.081690,145.391998,5282,10.0,U,Pacific/Port_Moresby,airport,OurAirports
1,2,Madang Airport,Madang,Papua New Guinea,MAG,AYMD,-5.207080,145.789001,20,10.0,U,Pacific/Port_Moresby,airport,OurAirports
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,AYMH,-5.826790,144.296005,5388,10.0,U,Pacific/Port_Moresby,airport,OurAirports
3,4,Nadzab Airport,Nadzab,Papua New Guinea,LAE,AYNZ,-6.569803,146.725977,239,10.0,U,Pacific/Port_Moresby,airport,OurAirports
4,5,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,AYPY,-9.443380,147.220001,146,10.0,U,Pacific/Port_Moresby,airport,OurAirports


In [8]:
airlines_raw.head()


,airline_id,name,alias,iata_code,icao_code,callsign,country_name,active
0,-1,Unknown,NaN,-,NaN,NaN,NaN,Y
1,1,Private flight,NaN,-,NaN,NaN,NaN,Y
2,2,135 Airways,NaN,NaN,GNL,GENERAL,United States,N
3,3,1Time Airline,NaN,1T,RNX,NEXTIME,South Africa,Y
4,4,2 Sqn No 1 Elementary Flying Training School,NaN,NaN,WYT,NaN,United Kingdom,N


In [9]:
planes_raw.head()


,name,iata_code,icao_code
0,Aerospatiale (Nord) 262,ND2,N262
1,Aerospatiale (Sud Aviation) Se.210 Caravelle,CRV,S210
2,Aerospatiale SN.601 Corvette,NDC,S601
3,Aerospatiale/Alenia ATR 42-300,AT4,AT43
4,Aerospatiale/Alenia ATR 42-500,AT5,AT45


In [10]:
routes_raw.head()


,airline_code,airline_id,source_airport_code,source_airport_id,destination_airport_code,destination_airport_id,codeshare,stops,equipment
0,2B,410.0,AER,2965.0,KZN,2990.0,NaN,0,CR2
1,2B,410.0,ASF,2966.0,KZN,2990.0,NaN,0,CR2
2,2B,410.0,ASF,2966.0,MRV,2962.0,NaN,0,CR2
3,2B,410.0,CEK,2968.0,KZN,2990.0,NaN,0,CR2
4,2B,410.0,CEK,2968.0,OVB,4078.0,NaN,0,CR2


## 5. Clean `Country`

The `Country` table uses `iso_code` as its primary key.

Using `iso_code` is cleaner than using the country name because country names may vary in spelling, punctuation or wording. For example, `"United States"`, `"USA"` and `"United States of America"` can all refer to the same country, but the ISO code is consistently `US`.


In [11]:
countries = countries_raw.copy()

countries["name"] = countries["name"].astype("string").str.strip()
countries["iso_code"] = countries["iso_code"].astype("string").str.strip().str.upper()
countries["dafif_code"] = countries["dafif_code"].astype("string").str.strip().str.upper()

# Keep only rows with a usable ISO code, since this is our primary key.
countries = countries.dropna(subset=["iso_code"])
countries = countries.drop_duplicates(subset=["iso_code"])

countries = countries[[
    "iso_code",
    "name",
    "dafif_code",
]]

countries.head()


,iso_code,name,dafif_code
0,BQ,"Bonaire, Saint Eustatius and Saba",<NA>
1,AW,Aruba,AA
2,AG,Antigua and Barbuda,AC
3,AE,United Arab Emirates,AE
4,AF,Afghanistan,AF


## 6. Clean `Airport`

The raw airport file stores the country as a country name.  
The ERD uses `country_iso_code` instead, so we join the airport data to the `Country` table.

Raw example:

```text
country_name = "United Kingdom"
```

Cleaned version:

```text
country_iso_code = "GB"
```


In [12]:
airports = airports_raw.copy()

# Optional project decision:
# Keep only actual airports, excluding train stations, ferry terminals and unknown locations.
airports = airports[airports["type"] == "airport"].copy()

# Convert numeric fields.
airports["airport_id"] = pd.to_numeric(airports["airport_id"], errors="coerce").astype("Int64")
airports["latitude"] = pd.to_numeric(airports["latitude"], errors="coerce")
airports["longitude"] = pd.to_numeric(airports["longitude"], errors="coerce")
airports["altitude"] = pd.to_numeric(airports["altitude"], errors="coerce").astype("Int64")
airports["timezone_offset"] = pd.to_numeric(airports["timezone_offset"], errors="coerce")

# Clean text fields.
for col in [
    "name",
    "city",
    "country_name",
    "iata_code",
    "icao_code",
    "dst",
    "tz_database_timezone",
    "type",
    "source",
]:
    airports[col] = airports[col].astype("string").str.strip()

airports["iata_code"] = airports["iata_code"].str.upper()
airports["icao_code"] = airports["icao_code"].str.upper()

# Map country names to ISO codes.
airports = airports.merge(
    countries[["iso_code", "name"]],
    left_on="country_name",
    right_on="name",
    how="left",
    suffixes=("", "_country"),
)

airports = airports.rename(columns={"iso_code": "country_iso_code"})

# Save unmatched countries for review.
unmatched_airport_countries = airports[
    airports["country_iso_code"].isna()
][["airport_id", "name", "city", "country_name"]].drop_duplicates()

# Keep rows with valid primary keys.
airports = airports.dropna(subset=["airport_id"])
airports = airports.drop_duplicates(subset=["airport_id"])

airports = airports[[
    "airport_id",
    "name",
    "city",
    "country_iso_code",
    "iata_code",
    "icao_code",
    "latitude",
    "longitude",
    "altitude",
    "timezone_offset",
    "dst",
    "tz_database_timezone",
    "type",
    "source",
]]

airports.head()


,airport_id,name,city,country_iso_code,iata_code,icao_code,latitude,longitude,altitude,timezone_offset,dst,tz_database_timezone,type,source
0,1,Goroka Airport,Goroka,PG,GKA,AYGA,-6.081690,145.391998,5282,10.0,U,Pacific/Port_Moresby,airport,OurAirports
1,2,Madang Airport,Madang,PG,MAG,AYMD,-5.207080,145.789001,20,10.0,U,Pacific/Port_Moresby,airport,OurAirports
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,PG,HGU,AYMH,-5.826790,144.296005,5388,10.0,U,Pacific/Port_Moresby,airport,OurAirports
3,4,Nadzab Airport,Nadzab,PG,LAE,AYNZ,-6.569803,146.725977,239,10.0,U,Pacific/Port_Moresby,airport,OurAirports
4,5,Port Moresby Jacksons International Airport,Port Moresby,PG,POM,AYPY,-9.443380,147.220001,146,10.0,U,Pacific/Port_Moresby,airport,OurAirports


In [13]:
print(f"Unmatched airport countries: {len(unmatched_airport_countries)}")
unmatched_airport_countries.head()


Unmatched airport countries: 168


,airport_id,name,city,country_name
616,625,Vagar Airport,Vagar,Faroe Islands
863,883,Maya-Maya Airport,Brazzaville,Congo (Brazzaville)
864,884,Owando Airport,Owando,Congo (Kinshasa)
865,885,Ouesso Airport,Ouesso,Congo (Kinshasa)
866,886,Pointe Noire Airport,Pointe-noire,Congo (Brazzaville)


## 7. Clean `Airline`

As with airports, the raw airline file stores the country name.  
We map it to `country_iso_code` so that the `Airline` table can reference the `Country` table properly.


In [14]:
airlines = airlines_raw.copy()

airlines["airline_id"] = pd.to_numeric(airlines["airline_id"], errors="coerce").astype("Int64")

for col in [
    "name",
    "alias",
    "iata_code",
    "icao_code",
    "callsign",
    "country_name",
    "active",
]:
    airlines[col] = airlines[col].astype("string").str.strip()

airlines["iata_code"] = airlines["iata_code"].str.upper()
airlines["icao_code"] = airlines["icao_code"].str.upper()
airlines["active"] = airlines["active"].str.upper()

# Map country names to ISO codes.
airlines = airlines.merge(
    countries[["iso_code", "name"]],
    left_on="country_name",
    right_on="name",
    how="left",
    suffixes=("", "_country"),
)

airlines = airlines.rename(columns={"iso_code": "country_iso_code"})

# Save unmatched countries for review.
unmatched_airline_countries = airlines[
    airlines["country_iso_code"].isna()
][["airline_id", "name", "country_name"]].drop_duplicates()

# Keep rows with valid primary keys.
airlines = airlines.dropna(subset=["airline_id"])
airlines = airlines.drop_duplicates(subset=["airline_id"])

airlines = airlines[[
    "airline_id",
    "name",
    "alias",
    "iata_code",
    "icao_code",
    "callsign",
    "country_iso_code",
    "active",
]]

airlines.head()


,airline_id,name,alias,iata_code,icao_code,callsign,country_iso_code,active
0,-1,Unknown,<NA>,-,<NA>,<NA>,<NA>,Y
1,1,Private flight,<NA>,-,<NA>,<NA>,<NA>,Y
2,2,135 Airways,<NA>,<NA>,GNL,GENERAL,US,N
3,3,1Time Airline,<NA>,1T,RNX,NEXTIME,ZA,Y
4,4,2 Sqn No 1 Elementary Flying Training School,<NA>,<NA>,WYT,<NA>,GB,N


In [15]:
print(f"Unmatched airline countries: {len(unmatched_airline_countries)}")
unmatched_airline_countries.head()


Unmatched airline countries: 245


,airline_id,name,country_name
0,-1,Unknown,<NA>
1,1,Private flight,<NA>
28,28,Asiana Airlines,Republic of Korea
34,34,Afric'air Express,Ivory Coast
38,38,African Business and Transportations,Democratic Republic of the Congo


## 8. Clean `Plane`

The `Plane` table represents aircraft types.

The route equipment field uses IATA aircraft type codes, so we use `iata_code` as the primary key for `Plane`.


In [16]:
planes = planes_raw.copy()

for col in ["name", "iata_code", "icao_code"]:
    planes[col] = planes[col].astype("string").str.strip()

planes["iata_code"] = planes["iata_code"].str.upper()
planes["icao_code"] = planes["icao_code"].str.upper()

# Since route equipment uses IATA aircraft codes, require iata_code.
planes = planes.dropna(subset=["iata_code"])
planes = planes.drop_duplicates(subset=["iata_code"])

planes = planes[[
    "iata_code",
    "icao_code",
    "name",
]]

planes.head()


,iata_code,icao_code,name
0,ND2,N262,Aerospatiale (Nord) 262
1,CRV,S210,Aerospatiale (Sud Aviation) Se.210 Caravelle
2,NDC,S601,Aerospatiale SN.601 Corvette
3,AT4,AT43,Aerospatiale/Alenia ATR 42-300
4,AT5,AT45,Aerospatiale/Alenia ATR 42-500


## 9. Clean `Route`

The raw `routes.dat` file does not contain a route ID.  
To make a proper relational table, we create our own surrogate `route_id`.

The `Route` table references:

- `Airline.airline_id`
- `Airport.airport_id` as the source airport
- `Airport.airport_id` as the destination airport

Routes with missing or invalid airline/airport IDs are separated for review rather than imported into the final clean route table.


In [17]:
routes = routes_raw.copy()

routes["airline_id"] = pd.to_numeric(routes["airline_id"], errors="coerce").astype("Int64")
routes["source_airport_id"] = pd.to_numeric(routes["source_airport_id"], errors="coerce").astype("Int64")
routes["destination_airport_id"] = pd.to_numeric(routes["destination_airport_id"], errors="coerce").astype("Int64")
routes["stops"] = pd.to_numeric(routes["stops"], errors="coerce").astype("Int64")

for col in [
    "airline_code",
    "source_airport_code",
    "destination_airport_code",
    "codeshare",
    "equipment",
]:
    routes[col] = routes[col].astype("string").str.strip()

routes["airline_code"] = routes["airline_code"].str.upper()
routes["source_airport_code"] = routes["source_airport_code"].str.upper()
routes["destination_airport_code"] = routes["destination_airport_code"].str.upper()
routes["codeshare"] = routes["codeshare"].str.upper()

valid_airline_ids = set(airlines["airline_id"].dropna())
valid_airport_ids = set(airports["airport_id"].dropna())

valid_route_mask = (
    routes["airline_id"].isin(valid_airline_ids)
    & routes["source_airport_id"].isin(valid_airport_ids)
    & routes["destination_airport_id"].isin(valid_airport_ids)
)

rejected_routes = routes[~valid_route_mask].copy()
routes = routes[valid_route_mask].copy()

# Remove exact duplicate route records before assigning route_id.
routes = routes.drop_duplicates(subset=[
    "airline_id",
    "source_airport_id",
    "destination_airport_id",
    "codeshare",
    "stops",
    "equipment",
])

# Create a surrogate route_id because OpenFlights routes do not have one.
routes = routes.reset_index(drop=True)
routes.insert(0, "route_id", routes.index + 1)

routes_clean = routes[[
    "route_id",
    "airline_id",
    "source_airport_id",
    "destination_airport_id",
    "codeshare",
    "stops",
]]

routes_clean.head()


,route_id,airline_id,source_airport_id,destination_airport_id,codeshare,stops
0,1,410,2965,2990,<NA>,0
1,2,410,2966,2990,<NA>,0
2,3,410,2966,2962,<NA>,0
3,4,410,2968,2990,<NA>,0
4,5,410,2968,4078,<NA>,0


In [18]:
print(f"Rejected routes because of invalid/missing foreign keys: {len(rejected_routes)}")
rejected_routes.head()


Rejected routes because of invalid/missing foreign keys: 1347


,airline_code,airline_id,source_airport_code,source_airport_id,destination_airport_code,destination_airport_id,codeshare,stops,equipment
7,2B,410,DME,4029,TGK,<NA>,<NA>,0,CR2
38,2B,410,TGK,<NA>,DME,4029,<NA>,0,CR2
48,2G,1654,IKT,2937,KCK,<NA>,<NA>,0,AN4
54,2G,1654,KCK,<NA>,IKT,2937,<NA>,0,AN4
170,2O,146,ADQ,3531,AOS,7167,<NA>,0,BNI


## 10. Create `RouteEquipment`

The raw `equipment` column is the messiest part of the route data.

For example, one route might have:

```text
744 777
```

That means the route may use both aircraft type `744` and aircraft type `777`.

This is a multi-valued field, so it should not remain inside the `Route` table.  
Instead, we split it into a bridge table called `RouteEquipment`.

This resolves the many-to-many relationship:

```text
Route many-to-many Plane
```


In [19]:
# Show examples of raw equipment values.
routes[["route_id", "equipment"]].dropna().head(10)


,route_id,equipment
0,1,CR2
1,2,CR2
2,3,CR2
3,4,CR2
4,5,CR2
5,6,CR2
6,7,CR2
7,8,CR2
8,9,CR2
9,10,CR2


In [20]:
route_equipment = routes[["route_id", "equipment"]].dropna().copy()

# Equipment is space-separated, e.g. "744 777".
route_equipment["plane_iata_code"] = route_equipment["equipment"].str.split()

# Split each list into separate rows.
route_equipment = route_equipment.explode("plane_iata_code")

route_equipment["plane_iata_code"] = (
    route_equipment["plane_iata_code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Separate equipment codes that do not appear in the plane table.
valid_plane_codes = set(planes["iata_code"].dropna())

unmatched_equipment_codes = route_equipment[
    ~route_equipment["plane_iata_code"].isin(valid_plane_codes)
][["route_id", "plane_iata_code"]].drop_duplicates()

route_equipment = route_equipment[
    route_equipment["plane_iata_code"].isin(valid_plane_codes)
].copy()

route_equipment = route_equipment[[
    "route_id",
    "plane_iata_code",
]].drop_duplicates()

route_equipment.head()


,route_id,plane_iata_code
0,1,CR2
1,2,CR2
2,3,CR2
3,4,CR2
4,5,CR2


In [21]:
print(f"Unmatched equipment code rows: {len(unmatched_equipment_codes)}")
unmatched_equipment_codes.head()


Unmatched equipment code rows: 14066


,route_id,plane_iata_code
77,78,CRJ
78,79,CRJ
79,80,CRJ
82,83,CRJ
83,84,CRJ


## 11. Compare before and after

Before cleaning, a route could store multiple aircraft types in one field:

```text
route_id | equipment
1        | 744 777
```

After cleaning, each route-aircraft pairing gets its own row:

```text
route_id | plane_iata_code
1        | 744
1        | 777
```


In [22]:
# Pick a route with multiple equipment codes, if one exists.
multi_equipment = routes[
    routes["equipment"].fillna("").str.contains(" ", regex=False)
][["route_id", "equipment"]].head(1)

multi_equipment


,route_id,equipment
60,61,142 141


In [23]:
if not multi_equipment.empty:
    example_route_id = multi_equipment.iloc[0]["route_id"]
    display(route_equipment[route_equipment["route_id"] == example_route_id])
else:
    print("No route with multiple equipment codes found.")


,route_id,plane_iata_code
60,61,142
60,61,141


## 12. Check foreign key relationships

Before importing into SQL and enforcing foreign keys, we check that the cleaned tables line up with the ERD.

These checks should ideally return zero missing references (this validation code is from ChatGPT so please use this part with extreme caution).


In [24]:
missing_airline_refs = set(routes_clean["airline_id"].dropna()) - set(airlines["airline_id"].dropna())

missing_source_airport_refs = (
    set(routes_clean["source_airport_id"].dropna()) - set(airports["airport_id"].dropna())
)

missing_destination_airport_refs = (
    set(routes_clean["destination_airport_id"].dropna()) - set(airports["airport_id"].dropna())
)

missing_plane_refs = (
    set(route_equipment["plane_iata_code"].dropna()) - set(planes["iata_code"].dropna())
)

missing_airport_country_refs = (
    set(airports["country_iso_code"].dropna()) - set(countries["iso_code"].dropna())
)

missing_airline_country_refs = (
    set(airlines["country_iso_code"].dropna()) - set(countries["iso_code"].dropna())
)

foreign_key_checks = pd.DataFrame({
    "Relationship": [
        "routes.airline_id -> airlines.airline_id",
        "routes.source_airport_id -> airports.airport_id",
        "routes.destination_airport_id -> airports.airport_id",
        "route_equipment.plane_iata_code -> planes.iata_code",
        "airports.country_iso_code -> countries.iso_code",
        "airlines.country_iso_code -> countries.iso_code",
    ],
    "Missing references": [
        len(missing_airline_refs),
        len(missing_source_airport_refs),
        len(missing_destination_airport_refs),
        len(missing_plane_refs),
        len(missing_airport_country_refs),
        len(missing_airline_country_refs),
    ]
})

foreign_key_checks


,Relationship,Missing references
0,routes.airline_id -> airlines.airline_id,0
1,routes.source_airport_id -> airports.airport_id,0
2,routes.destination_airport_id -> airports.airport_id,0
3,route_equipment.plane_iata_code -> planes.iata_code,0
4,airports.country_iso_code -> countries.iso_code,0
5,airlines.country_iso_code -> countries.iso_code,0


## 13. Summary of cleaned tables

This summary shows how many rows each cleaned table contains.


In [25]:
summary = pd.DataFrame({
    "Table": [
        "countries",
        "airports",
        "airlines",
        "planes",
        "routes",
        "route_equipment",
    ],
    "Rows": [
        len(countries),
        len(airports),
        len(airlines),
        len(planes),
        len(routes_clean),
        len(route_equipment),
    ],
})

summary


,Table,Rows
0,countries,239
1,airports,7698
2,airlines,6162
3,planes,220
4,routes,66316
5,route_equipment,77669


## 14. Export cleaned CSV files

Each cleaned CSV corresponds to one table in the ERD.

Recommended SQL import order:

1. `countries`
2. `airports`
3. `airlines`
4. `planes`
5. `routes`
6. `route_equipment`

This order matters because the child tables depend on the parent tables through foreign keys (at least this broke when executed in the wrong order, c.f. with ERD).


In [26]:
countries.to_csv(CLEANED / "countries.csv", index=False)
airports.to_csv(CLEANED / "airports.csv", index=False)
airlines.to_csv(CLEANED / "airlines.csv", index=False)
planes.to_csv(CLEANED / "planes.csv", index=False)
routes_clean.to_csv(CLEANED / "routes.csv", index=False)
route_equipment.to_csv(CLEANED / "route_equipment.csv", index=False)

# Also export rejected/unmatched rows for transparency about what was dropped.
unmatched_airport_countries.to_csv(CLEANED / "unmatched_airport_countries.csv", index=False)
unmatched_airline_countries.to_csv(CLEANED / "unmatched_airline_countries.csv", index=False)
rejected_routes.to_csv(CLEANED / "rejected_routes.csv", index=False)
unmatched_equipment_codes.to_csv(CLEANED / "unmatched_equipment_codes.csv", index=False)

print("Cleaned CSV files exported to:", CLEANED.resolve())


Cleaned CSV files exported to: C:\Users\maraj\Documents\SE-ADV-DATA-JUNE\SQL-Training\Openflights-project\cleaned


## 15. Final cleaning decisions

The main cleaning decisions were:

| Raw issue | Cleaning decision |
|---|---|
| `.dat` files had no headers | Added column names manually |
| Missing values appeared as `\N` | Converted them into null values |
| Countries appeared as names in airport and airline data | Mapped names to ISO country codes |
| Airports included non-airport locations | Kept only rows where `type = "airport"` |
| Routes had no primary key | Created a surrogate `route_id` |
| Route `equipment` field contained multiple aircraft codes | Split it into `RouteEquipment` |
| Some foreign key references were invalid or missing | Separated those rows for review |
| Some aircraft equipment codes did not match `Plane` | Separated unmatched codes for review |

The most important database design change was splitting `equipment` into a bridge table, because this turns a multi-valued field into a proper many-to-many relationship between `Route` and `Plane`.


## 16. Mermaid ERD used for the cleaned data

You can paste this into Mermaid Live Editor or a Markdown file that supports Mermaid.

```mermaid
erDiagram

    COUNTRY ||--o{ AIRPORT : contains
    COUNTRY ||--o{ AIRLINE : registers

    AIRLINE ||--o{ ROUTE : operates

    AIRPORT ||--o{ ROUTE : "source for"
    AIRPORT ||--o{ ROUTE : "destination for"

    ROUTE ||--o{ ROUTE_EQUIPMENT : has
    PLANE ||--o{ ROUTE_EQUIPMENT : used_as

    COUNTRY {
        string iso_code PK
        string name
        string dafif_code
    }

    AIRPORT {
        int airport_id PK
        string name
        string city
        string country_iso_code FK
        string iata_code
        string icao_code
        float latitude
        float longitude
        int altitude
        float timezone_offset
        string dst
        string tz_database_timezone
        string type
        string source
    }

    AIRLINE {
        int airline_id PK
        string name
        string alias
        string iata_code
        string icao_code
        string callsign
        string country_iso_code FK
        string active
    }

    ROUTE {
        int route_id PK
        int airline_id FK
        int source_airport_id FK
        int destination_airport_id FK
        string codeshare
        int stops
    }

    PLANE {
        string iata_code PK
        string icao_code
        string name
    }

    ROUTE_EQUIPMENT {
        int route_id PK, FK
        string plane_iata_code PK, FK
    }
```
